Instalar paquete de libreria

In [1]:
!pip install google-cloud-compute

     ---------------------------------------- 3.8/3.8 MB 20.4 MB/s eta 0:00:00


You should consider upgrading via the 'C:\Users\e.rangel\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip' command.


Necesitamos autenticarnos para darle acceso a Colab a nuestra cuenta de GCP

In [2]:
from google.colab import auth
auth.authenticate_user()

ModuleNotFoundError: No module named 'google.colab'

Importamos las librerías necesarias

In [3]:
from __future__ import annotations
from collections.abc import Iterable
from google.cloud import compute_v1

In [5]:
par_project_id = 'project-d0c76d9f-d27a-4298-8e3'

Crear instancia

In [6]:
def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def create_instance(
    project_id: str, zone: str, instance_name: str, machine_type: str, source_image_project: str, source_image_family: str, network_name: str
) -> None:
    instance_client = compute_v1.InstancesClient()

    # Configure the machine type
    machine_type_full_path = f"zones/{zone}/machineTypes/{machine_type}"

    # Create the boot disk with the correct source image
    disk = compute_v1.AttachedDisk(
        initialize_params=compute_v1.AttachedDiskInitializeParams(
            source_image=f"projects/{source_image_project}/global/images/family/{source_image_family}"
        ),
        auto_delete=True,
        boot=True,
        type_=compute_v1.AttachedDisk.Type.PERSISTENT.name,
    )

    # Configure the network interface
    network_interface = compute_v1.NetworkInterface(name=network_name)

    # Define the instance resource
    instance = compute_v1.Instance(
        name=instance_name,
        machine_type=machine_type_full_path,
        disks=[disk],
        network_interfaces=[network_interface],
    )

    print(f"Creating instance {instance_name} in {zone}...")
    operation = instance_client.insert(project=project_id, zone=zone, instance_resource=instance)
    wait_for_extended_operation(operation, "instance creation")
    print(f"Instance {instance_name} created.")


In [13]:
create_instance(
    project_id=par_project_id,
    zone="us-central1-f",
    instance_name="sd-colab-vm-02",
    # machine_type="n1-standard-1",
    machine_type="e2-medium",
    source_image_project="debian-cloud",  # Project hosting the image
    source_image_family="debian-11",      # Updated to Debian 11
    network_name="global/networks/default"
)

Creating instance sd-colab-vm-02 in us-central1-f...
Instance sd-colab-vm-02 created.


Listar las instancias de Compute Engine por Zona

In [14]:
def list_instances(project_id: str, zone: str) -> Iterable[compute_v1.Instance]:
    instance_client = compute_v1.InstancesClient()
    instance_list = instance_client.list(project=project_id, zone=zone)

    print(f"Instances found in zone {zone}:")
    for instance in instance_list:
        print(f" - {instance.name} ({instance.machine_type})")

    return instance_list

In [15]:
instancias = list_instances(par_project_id,'us-central1-f')

Instances found in zone us-central1-f:
 - sd-colab-vm-02 (https://www.googleapis.com/compute/v1/projects/project-d0c76d9f-d27a-4298-8e3/zones/us-central1-f/machineTypes/e2-medium)
 - sd-consola-igroup-01-zjrk (https://www.googleapis.com/compute/v1/projects/project-d0c76d9f-d27a-4298-8e3/zones/us-central1-f/machineTypes/e2-medium)


In [16]:
lista_instancias = list(instancias)

for instance in lista_instancias:
    with open("my_file.txt", "w") as archivo:
    # Escribimos los datos en el archivo
        archivo.write(str(instance))

In [17]:
!pwd

'pwd' is not recognized as an internal or external command,
operable program or batch file.


Detener una instancia de Compute Engine

In [18]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def stop_instance(project_id: str, zone: str, instance_name: str) -> None:
    instance_client = compute_v1.InstancesClient()

    operation = instance_client.stop(
        project=project_id, zone=zone, instance=instance_name
    )
    wait_for_extended_operation(operation, "instance stopping")


In [ ]:
stop_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-02')

Iniciar una instancia de Compute Engine

In [21]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def start_instance(project_id: str, zone: str, instance_name: str) -> None:

    instance_client = compute_v1.InstancesClient()

    operation = instance_client.start(
        project=project_id, zone=zone, instance=instance_name
    )

    wait_for_extended_operation(operation, "instance start")


In [22]:
start_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-02')

Elimina una instancia de Compute Engine

In [23]:
from __future__ import annotations

import sys
from typing import Any

from google.api_core.extended_operation import ExtendedOperation
from google.cloud import compute_v1

def wait_for_extended_operation(
    operation: ExtendedOperation, verbose_name: str = "operation", timeout: int = 300
) -> Any:

    result = operation.result(timeout=timeout)

    if operation.error_code:
        print(
            f"Error during {verbose_name}: [Code: {operation.error_code}]: {operation.error_message}",
            file=sys.stderr,
            flush=True,
        )
        print(f"Operation ID: {operation.name}", file=sys.stderr, flush=True)
        raise operation.exception() or RuntimeError(operation.error_message)

    if operation.warnings:
        print(f"Warnings during {verbose_name}:\n", file=sys.stderr, flush=True)
        for warning in operation.warnings:
            print(f" - {warning.code}: {warning.message}", file=sys.stderr, flush=True)

    return result

def delete_instance(project_id: str, zone: str, machine_name: str) -> None:

    instance_client = compute_v1.InstancesClient()

    print(f"Deleting {machine_name} from {zone}...")
    operation = instance_client.delete(
        project=project_id, zone=zone, instance=machine_name
    )
    wait_for_extended_operation(operation, "instance deletion")
    print(f"Instance {machine_name} deleted.")


In [24]:
delete_instance(par_project_id, 'us-central1-f', 'sd-colab-vm-02')

Deleting sd-colab-vm-02 from us-central1-f...
Instance sd-colab-vm-02 deleted.


In [25]:
from google.cloud import compute_v1

project_id = par_project_id
zone = "us-central1-f"
instance_name = "lab06-vm"

client = compute_v1.InstancesClient()

instance = client.get(
    project=project_id,
    zone=zone,
    instance=instance_name
)

external_ip = instance.network_interfaces[0].access_configs[0].nat_i_p
internal_ip = instance.network_interfaces[0].network_i_p

print("External IP:", external_ip)
print("Internal IP:", internal_ip)

External IP: 34.9.173.85
Internal IP: 10.128.0.18
